# Feature Engineering Analysis
## Custom Pakistani Dataset

This notebook analyzes extracted facial features:
- Eye Aspect Ratio (EAR)
- Mouth Aspect Ratio (MAR)
- Head Pose (pitch, yaw, roll)
- Occlusion Detection

**Purpose:** Generate statistics and visualizations for FYP report

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Styling
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

print("✅ Imports successful")

## 1. Load Feature Data

In [ ]:
# Load extracted features
features_path = Path('datasets/features/custom_pakistani_features.csv')

if features_path.exists():
    df = pd.read_csv(features_path)
    print(f"✅ Loaded {len(df)} samples")
    print(f"\n📊 Class distribution:")
    print(df['class'].value_counts())
    
    # Display first few rows
    display(df.head())
else:
    print("❌ Features file not found!")
    print(f"   Expected: {features_path}")
    print("\n   Run feature extraction first:")
    print("   python process_custom_dataset.py")

## 2. Statistical Summary

In [ ]:
# Overall statistics
print("="*70)
print("OVERALL FEATURE STATISTICS")
print("="*70)

feature_cols = ['avg_ear', 'mar', 'head_pitch', 'head_yaw', 'head_roll', 
                'face_visibility_score']

display(df[feature_cols].describe())

# Per-class statistics
print("\n" + "="*70)
print("PER-CLASS STATISTICS")
print("="*70)

for class_name in ['ALERT', 'DROWSY', 'DISTRACTED']:
    print(f"\n{class_name}:")
    class_data = df[df['class'] == class_name][feature_cols]
    display(class_data.describe().T[['mean', 'std', 'min', 'max']])

## 3. Feature Distributions (For Report!)

In [ ]:
# Create comprehensive distribution plots
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 1. EAR Distribution
sns.violinplot(data=df, x='class', y='avg_ear', ax=axes[0,0], palette='Set2')
axes[0,0].set_title('Eye Aspect Ratio (EAR) by Class', fontsize=14, fontweight='bold')
axes[0,0].axhline(y=0.25, color='green', linestyle='--', label='Alert threshold', linewidth=2)
axes[0,0].axhline(y=0.20, color='red', linestyle='--', label='Drowsy threshold', linewidth=2)
axes[0,0].set_ylabel('EAR', fontsize=12)
axes[0,0].legend()
axes[0,0].grid(alpha=0.3)

# 2. MAR Distribution
sns.violinplot(data=df, x='class', y='mar', ax=axes[0,1], palette='Set2')
axes[0,1].set_title('Mouth Aspect Ratio (MAR) by Class', fontsize=14, fontweight='bold')
axes[0,1].axhline(y=0.5, color='orange', linestyle='--', label='Normal', linewidth=2)
axes[0,1].axhline(y=0.6, color='red', linestyle='--', label='Yawn threshold', linewidth=2)
axes[0,1].set_ylabel('MAR', fontsize=12)
axes[0,1].legend()
axes[0,1].grid(alpha=0.3)

# 3. Head Yaw (Distraction)
sns.violinplot(data=df, x='class', y='head_yaw', ax=axes[0,2], palette='Set2')
axes[0,2].set_title('Head Yaw Angle by Class', fontsize=14, fontweight='bold')
axes[0,2].axhline(y=30, color='red', linestyle='--', label='Distraction threshold', linewidth=2)
axes[0,2].axhline(y=-30, color='red', linestyle='--', linewidth=2)
axes[0,2].set_ylabel('Yaw (degrees)', fontsize=12)
axes[0,2].legend()
axes[0,2].grid(alpha=0.3)

# 4. Head Pitch
sns.violinplot(data=df, x='class', y='head_pitch', ax=axes[1,0], palette='Set2')
axes[1,0].set_title('Head Pitch Angle by Class', fontsize=14, fontweight='bold')
axes[1,0].axhline(y=20, color='orange', linestyle='--', label='Looking down', linewidth=2)
axes[1,0].set_ylabel('Pitch (degrees)', fontsize=12)
axes[1,0].legend()
axes[1,0].grid(alpha=0.3)

# 5. Face Visibility (Occlusion)
sns.boxplot(data=df, x='class', y='face_visibility_score', ax=axes[1,1], palette='Set2')
axes[1,1].set_title('Face Visibility Score by Class', fontsize=14, fontweight='bold')
axes[1,1].axhline(y=0.6, color='red', linestyle='--', label='Occlusion threshold', linewidth=2)
axes[1,1].set_ylabel('Visibility Score (0-1)', fontsize=12)
axes[1,1].legend()
axes[1,1].grid(alpha=0.3)

# 6. Combined EAR and MAR scatter
for class_name in ['ALERT', 'DROWSY', 'DISTRACTED']:
    class_data = df[df['class'] == class_name]
    axes[1,2].scatter(class_data['avg_ear'], class_data['mar'], 
                     label=class_name, alpha=0.6, s=30)
axes[1,2].set_xlabel('EAR', fontsize=12)
axes[1,2].set_ylabel('MAR', fontsize=12)
axes[1,2].set_title('EAR vs MAR (Class Separation)', fontsize=14, fontweight='bold')
axes[1,2].axvline(x=0.20, color='red', linestyle='--', alpha=0.5)
axes[1,2].axhline(y=0.6, color='orange', linestyle='--', alpha=0.5)
axes[1,2].legend()
axes[1,2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('reports/feature_distributions.png', dpi=300, bbox_inches='tight')
print("✅ Saved: reports/feature_distributions.png")
plt.show()

## 4. Correlation Analysis

In [ ]:
# Feature correlation matrix
correlation = df[feature_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0, 
            fmt='.2f', linewidths=1, square=True,
            cbar_kws={'label': 'Correlation Coefficient'})
plt.title('Feature Correlation Matrix', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('reports/feature_correlation.png', dpi=300, bbox_inches='tight')
print("✅ Saved: reports/feature_correlation.png")
plt.show()

# Print strong correlations
print("\n🔍 Strong Correlations (|r| > 0.5):")
for i in range(len(correlation.columns)):
    for j in range(i+1, len(correlation.columns)):
        if abs(correlation.iloc[i, j]) > 0.5:
            print(f"   {correlation.columns[i]} ↔ {correlation.columns[j]}: {correlation.iloc[i, j]:.3f}")

## 5. Occlusion Analysis (FALSE POSITIVE HANDLING)

In [ ]:
# Occlusion statistics
print("="*70)
print("OCCLUSION ANALYSIS (Critical for Pakistani Context!)")
print("="*70)

# Overall occlusion rates
print("\n📊 Overall Occlusion Rates:")
print(f"   Eyes occluded: {df['eyes_occluded'].sum()} / {len(df)} ({df['eyes_occluded'].mean()*100:.1f}%)")
print(f"   Mouth occluded: {df['mouth_occluded'].sum()} / {len(df)} ({df['mouth_occluded'].mean()*100:.1f}%)")

# Per-class occlusion rates
print("\n📊 Occlusion Rates by Class:")
occlusion_by_class = df.groupby('class')[['eyes_occluded', 'mouth_occluded']].mean() * 100
display(occlusion_by_class)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Eyes occlusion
occlusion_data = df.groupby('class')['eyes_occluded'].value_counts(normalize=True).unstack() * 100
occlusion_data.plot(kind='bar', stacked=True, ax=axes[0], color=['lightgreen', 'salmon'])
axes[0].set_title('Eyes Occlusion Rate by Class', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Percentage (%)', fontsize=12)
axes[0].set_xlabel('Class', fontsize=12)
axes[0].legend(['Visible', 'Occluded'], loc='upper right')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

# Mouth occlusion
occlusion_data = df.groupby('class')['mouth_occluded'].value_counts(normalize=True).unstack() * 100
occlusion_data.plot(kind='bar', stacked=True, ax=axes[1], color=['lightgreen', 'salmon'])
axes[1].set_title('Mouth Occlusion Rate by Class', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Percentage (%)', fontsize=12)
axes[1].set_xlabel('Class', fontsize=12)
axes[1].legend(['Visible', 'Occluded'], loc='upper right')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.savefig('reports/occlusion_analysis.png', dpi=300, bbox_inches='tight')
print("\n✅ Saved: reports/occlusion_analysis.png")
plt.show()

## 6. Feature Importance (Classification)

In [ ]:
# Use Random Forest to determine feature importance
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# Prepare data
X = df[feature_cols]
y = df['class']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Evaluate
y_pred = rf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("="*70)
print("BASELINE CLASSIFICATION USING EXTRACTED FEATURES")
print("="*70)
print(f"\nAccuracy: {accuracy*100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("\n📊 Feature Importance:")
display(feature_importance)

# Visualize
plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance, y='feature', x='importance', palette='viridis')
plt.title('Feature Importance for Classification', fontsize=16, fontweight='bold')
plt.xlabel('Importance Score', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('reports/feature_importance.png', dpi=300, bbox_inches='tight')
print("\n✅ Saved: reports/feature_importance.png")
plt.show()

## 7. Summary Statistics for Report

In [ ]:
# Generate comprehensive summary
print("="*70)
print("SUMMARY FOR FYP REPORT")
print("="*70)

summary = {}

for class_name in ['ALERT', 'DROWSY', 'DISTRACTED']:
    class_data = df[df['class'] == class_name]
    
    summary[class_name] = {
        'count': len(class_data),
        'avg_ear_mean': class_data['avg_ear'].mean(),
        'avg_ear_std': class_data['avg_ear'].std(),
        'mar_mean': class_data['mar'].mean(),
        'mar_std': class_data['mar'].std(),
        'head_yaw_mean': abs(class_data['head_yaw']).mean(),
        'head_yaw_std': class_data['head_yaw'].std(),
        'face_visibility_mean': class_data['face_visibility_score'].mean(),
        'eyes_occluded_pct': class_data['eyes_occluded'].mean() * 100,
        'mouth_occluded_pct': class_data['mouth_occluded'].mean() * 100,
    }

summary_df = pd.DataFrame(summary).T
display(summary_df)

# Save summary
summary_df.to_csv('reports/feature_summary_statistics.csv')
print("\n✅ Saved: reports/feature_summary_statistics.csv")

print("\n" + "="*70)
print("KEY FINDINGS FOR REPORT:")
print("="*70)
print(f"\n1. EAR (Eye Aspect Ratio):")
print(f"   - ALERT: {summary['ALERT']['avg_ear_mean']:.3f} ± {summary['ALERT']['avg_ear_std']:.3f}")
print(f"   - DROWSY: {summary['DROWSY']['avg_ear_mean']:.3f} ± {summary['DROWSY']['avg_ear_std']:.3f}")
print(f"   - Difference: {(summary['ALERT']['avg_ear_mean'] - summary['DROWSY']['avg_ear_mean']):.3f}")

print(f"\n2. MAR (Mouth Aspect Ratio):")
print(f"   - ALERT: {summary['ALERT']['mar_mean']:.3f} ± {summary['ALERT']['mar_std']:.3f}")
print(f"   - DROWSY: {summary['DROWSY']['mar_mean']:.3f} ± {summary['DROWSY']['mar_std']:.3f}")

print(f"\n3. Head Yaw (Distraction):")
print(f"   - ALERT: {summary['ALERT']['head_yaw_mean']:.1f}° ± {summary['ALERT']['head_yaw_std']:.1f}°")
print(f"   - DISTRACTED: {summary['DISTRACTED']['head_yaw_mean']:.1f}° ± {summary['DISTRACTED']['head_yaw_std']:.1f}°")

print(f"\n4. Occlusion (False Positive Handling):")
print(f"   - Eyes occluded (sunglasses): {df['eyes_occluded'].mean()*100:.1f}% of images")
print(f"   - Mouth occluded (shawl): {df['mouth_occluded'].mean()*100:.1f}% of images")
print(f"   - Successfully detected and handled ✅")

print(f"\n5. Classification Performance:")
print(f"   - Using only extracted features: {accuracy*100:.1f}% accuracy")
print(f"   - Top features: {', '.join(feature_importance.head(3)['feature'].values)}")

## 8. Export for Documentation

In [ ]:
# Create folder for report figures
reports_dir = Path('reports')
reports_dir.mkdir(exist_ok=True)

print("="*70)
print("ALL FIGURES SAVED TO reports/ FOLDER")
print("="*70)
print("\nGenerated files:")
print("  ✅ reports/feature_distributions.png")
print("  ✅ reports/feature_correlation.png")
print("  ✅ reports/occlusion_analysis.png")
print("  ✅ reports/feature_importance.png")
print("  ✅ reports/feature_summary_statistics.csv")
print("\nUse these figures in your FYP report!")
print("="*70)